In [1]:
from pyfaidx import Fasta
import torch
import pandas as pd
import numpy as np

In [2]:
FOLD = 0
input_dir = "/scratch1/smaruj/genomic_map_transformation"

In [3]:
df = pd.read_csv(f"{input_dir}/df_select_fold{FOLD}.tsv", sep="\t")

In [4]:
# for testing
df = df[:10]

In [5]:
df['target_chrom'] = df['chrom'].shift(-1)
df['target_start'] = df['start'].shift(-1)
df['target_end'] = df['end'].shift(-1)

In [6]:
# Fill last row with values from the first row
df.loc[df.index[-1], 'target_chrom'] = df.loc[df.index[0], 'chrom']
df.loc[df.index[-1], 'target_start'] = df.loc[df.index[0], 'start']
df.loc[df.index[-1], 'target_end'] = df.loc[df.index[0], 'end']

# Convert to int
df['target_start'] = df['target_start'].astype(int)
df['target_end'] = df['target_end'].astype(int)

In [7]:
df

,chrom,start,end,fold,target_chrom,target_start,target_end
0,chr5,63203328,64514048,fold0,chr3,138672128,139982848
1,chr3,138672128,139982848,fold0,chr5,43542528,44853248
2,chr5,43542528,44853248,fold0,chr3,115171328,116482048
3,chr3,115171328,116482048,fold0,chr7,61700096,63010816
4,chr7,61700096,63010816,fold0,chr4,141115392,142426112
5,chr4,141115392,142426112,fold0,chr3,140638208,141948928
6,chr3,140638208,141948928,fold0,chr5,57632768,58943488
7,chr5,57632768,58943488,fold0,chr6,65816576,67127296
8,chr6,65816576,67127296,fold0,chr4,124506112,125816832
9,chr4,124506112,125816832,fold0,chr5,63203328,64514048


In [8]:
genome = Fasta("/project/fudenber_735/genomes/mm10/mm10.fa")

In [9]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [10]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [11]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [12]:
from tangermeme.tools import fimo

In [13]:
def get_sequence(chrom, start, end):
    return str(genome[chrom][start:end].seq.upper())

In [15]:
def jaccard_index(set1, set2):
    if not set1 and not set2:
        return 1.0
    return len(set1 & set2) / len(set1 | set2)

In [16]:
def one_hot_encode(seq):
    mapping = {"A":0, "C":1, "G":2, "T":3}
    ohe = np.zeros((4, len(seq)), dtype=np.float32)
    for i, base in enumerate(seq):
        if base in mapping:
            ohe[mapping[base], i] = 1.0
    return ohe

In [26]:
def hits_to_site_set(hits_df, bin_size=BIN_SIZE):
    site_set = set()
    for _, row in hits_df.iterrows():
        start, end, strand = row['start'], row['end'], row['strand']
        center = (start + end) // 2
        binned_pos = round(center / bin_size) * bin_size
        site_set.add((binned_pos, strand))
    return site_set

In [32]:
df

,chrom,start,end,fold,target_chrom,target_start,target_end
0,chr5,63203328,64514048,fold0,chr3,138672128,139982848
1,chr3,138672128,139982848,fold0,chr5,43542528,44853248
2,chr5,43542528,44853248,fold0,chr3,115171328,116482048
3,chr3,115171328,116482048,fold0,chr7,61700096,63010816
4,chr7,61700096,63010816,fold0,chr4,141115392,142426112
5,chr4,141115392,142426112,fold0,chr3,140638208,141948928
6,chr3,140638208,141948928,fold0,chr5,57632768,58943488
7,chr5,57632768,58943488,fold0,chr6,65816576,67127296
8,chr6,65816576,67127296,fold0,chr4,124506112,125816832
9,chr4,124506112,125816832,fold0,chr5,63203328,64514048


In [36]:
df['Jaccard_init_opt'] = 0.0
df['Jaccard_opt_target'] = 0.0

In [37]:
for idx, row in df.iterrows():
    # original sequence
    ohe_seq = one_hot_encode(get_sequence(row["chrom"], row["start"], row["end"]))
    ohe_tensor = torch.tensor(ohe_seq, dtype=torch.float32).unsqueeze(0)
    
    orig_hits = fimo.fimo(
        motifs=motifs_dict,
        sequences=ohe_tensor,
        threshold=1e-4,
        reverse_complement=True
    )[0]
    
    orig_site_set = hits_to_site_set(orig_hits)
    
    # optimized seq
    opt_path = f"/scratch1/smaruj/genomic_map_transformation/results_fold0/{row['chrom']}_{row['start']}_{row['end']}_seq.pt"
    opt_tensor = torch.load(opt_path)  # shape (4, L)
    
    opt_hits = fimo.fimo(
        motifs=motifs_dict,
        sequences=opt_tensor,
        threshold=1e-4,
        reverse_complement=True
    )[0]
    
    opt_site_set = hits_to_site_set(opt_hits)
    
    # target sequence
    tgt_seq = one_hot_encode(get_sequence(row["target_chrom"], row["target_start"], row["target_end"]))
    tgt_tensor = torch.tensor(tgt_seq, dtype=torch.float32).unsqueeze(0)
    
    tgt_hits = fimo.fimo(
        motifs=motifs_dict,
        sequences=tgt_tensor,
        threshold=1e-4,
        reverse_complement=True
    )[0]
    
    tgt_site_set = hits_to_site_set(tgt_hits)
    
    jacc_init_opt = jaccard_index(orig_site_set, opt_site_set)
    jacc_opt_target = jaccard_index(opt_site_set, tgt_site_set)
    
    df.at[idx, 'Jaccard_init_opt'] = jaccard_index(orig_site_set, opt_site_set)
    df.at[idx, 'Jaccard_opt_target'] = jaccard_index(opt_site_set, tgt_site_set)

/tmp/SLURM_1508869/ipykernel_3654715/3029017538.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  opt_tensor = torch.load(opt_path)  # shape (4, L)
/tmp/SLURM_1508869/ipy

In [38]:
df.to_csv("/scratch1/smaruj/genomic_map_transformation/ctcf_jaccard_results.tsv", sep="\t", index=False)

,chrom,start,end,fold,target_chrom,target_start,target_end,Jaccard_init_opt,Jaccard_opt_target
0,chr5,63203328,64514048,fold0,chr3,138672128,139982848,0.286267,0.000000
1,chr3,138672128,139982848,fold0,chr5,43542528,44853248,0.260753,0.000000
2,chr5,43542528,44853248,fold0,chr3,115171328,116482048,0.266667,0.000000
3,chr3,115171328,116482048,fold0,chr7,61700096,63010816,0.252822,0.000000
4,chr7,61700096,63010816,fold0,chr4,141115392,142426112,0.237838,0.002260
5,chr4,141115392,142426112,fold0,chr3,140638208,141948928,0.301384,0.002821
6,chr3,140638208,141948928,fold0,chr5,57632768,58943488,0.240000,0.000000
7,chr5,57632768,58943488,fold0,chr6,65816576,67127296,0.198556,0.000000
8,chr6,65816576,67127296,fold0,chr4,124506112,125816832,0.221698,0.002439
9,chr4,124506112,125816832,fold0,chr5,63203328,64514048,0.260291,0.000000
